# Exercise 05 — A neural network that classifies

## What you are about to see

Every exercise so far built a *piece*: an activation and its derivative (Ex. 01), layers
composed out of engine ops (Ex. 02–04). None of them trained anything. **This is the first
time the pieces are assembled into a working classifier and actually trained on real
data** — and the first time you see what "a neural network solves a classification
problem" means, end to end:

```
datasets.load_adult   ->  X, y  (model-ready NumPy)
          |
     nn.Linear + relu + nn.Linear      the model — a small MLP
          |
     cross_entropy(logits, y)          one scalar: how wrong we are
          |
     loss.backward()                   fills every parameter's .grad
          |
     optim.Adam(...).step()            nudges each parameter downhill
```

There is nothing to fill in here. Read a section, run its cell, and check the claim it
makes. The whole notebook runs in about **8 seconds**.

The notation is the one from the lecture notes — inputs are **columns**, the input is
augmented with $x_0 = 1$ so the bias is row 0 of the weight matrix, and a layer is
$\mathbf{W}^{\top}\widetilde{\mathbf{X}}$. Each section below names the part of *Lecture I
— Matrix Modeling* it puts to work, so you meet every object twice: once on paper, once
running.

In [ ]:
# Run me first: make ``bert_cpu`` importable whether Jupyter was started in the
# project root or inside ``exercises/``, exactly like the scripts in this folder do.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "bert_cpu").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np

import datasets
from bert_cpu import engine as cpu
from bert_cpu import nn
from bert_cpu import optim
from bert_cpu.loss import cross_entropy

print("ready — engine imported from", ROOT)

## 1. The problem, and what counts as a good answer

The **UCI Adult** dataset: one row per person from a census extract, and the question is
whether that person's income exceeds 50K a year. Two classes, 32 561 rows to learn from
and 16 281 held back for the final test.

Each row reaches us as **108 numbers**, not as the raw census columns. The continuous
fields (age, hours per week, …) were **standardised**, $x' = (x - \mu)/s$, and the
categorical ones (occupation, education, …) **one-hot encoded**. That preprocessing is
exactly the argument of Lecture I, *Why Input Normalization or Standardization Is
Necessary*: the pre-activation is $z = \sum_i w_i x_i$, so a feature stored on a much
larger scale dominates the sum and drives the activation into saturation, where it stops
responding to anything. `datasets.load_adult` has already done this work.

**Fix a baseline before training anything.** About 76 % of the rows are `<=50K`, so a
model that ignores its input and always answers `<=50K` scores ≈ 0.76. That is the number
our classifier has to beat; an accuracy means nothing until you know what a dumb answer
scores.

Conventions, as everywhere in this library: features run down axis 0 and samples across
axis 1, so `X` is `(n_features, n_samples)` — one column is one person.

In [ ]:
cpu.set_seed(0)                             # reproducible init + shuffling

train_ds = datasets.load_adult("train")
test_ds = datasets.load_adult("test")
print(f"\nData: {train_ds}   {test_ds}")
print(f"Features per sample: {train_ds.n_features}  (standardised + one-hot)")

majority = 1.0 - train_ds.y.mean()          # share of the <=50K class
print(f"Majority-class baseline on train: {majority:.4f}")

## 2. The model: from one neuron to this network

The classifier is two dense layers with a ReLU between them:

$$\mathbf{Z}^{(1)} = \bigl(\mathbf{W}^{(1)}\bigr)^{\top}\widetilde{\mathbf{X}},
\qquad
\mathbf{A}^{(1)} = \mathrm{ReLU}\bigl(\mathbf{Z}^{(1)}\bigr),
\qquad
\mathbf{Z}^{(2)} = \bigl(\mathbf{W}^{(2)}\bigr)^{\top}\widetilde{\mathbf{A}}^{(1)}$$

with $\mathbf{W}^{(1)} \in \mathbb{R}^{109 \times 64}$ and
$\mathbf{W}^{(2)} \in \mathbb{R}^{65 \times 2}$. The extra row in each is the bias: the
input is augmented with $x_0 = 1$, so $w_0$ *is* the bias and a layer stays a single
matrix product (Lecture I, *From One Neuron to a Fully Connected Layer*, and *Batch
processing: many samples at once* — all 26 049 training columns go through that one
product at a time).

Two remarks worth pausing on:

- **Why the ReLU in the middle.** Without it, $(\mathbf{W}^{(2)})^{\top}(\mathbf{W}^{(1)})^{\top}\widetilde{\mathbf{X}}$
  is again a single linear map: two layers would buy nothing over one. The non-linearity is
  what lets the hidden layer bend the space (Lecture I, *Stacking Layers: the Multilayer
  Network*).
- **The hidden layer is a new description of the person.** 108 raw coordinates in, 64
  learned coordinates out, chosen by training to make the last layer's job easy — Lecture
  I, *Fully Connected Networks as a Change of Representation*.

In [ ]:
class AdultMLP(nn.Module):
    """``Linear -> ReLU -> Linear`` classifier over the Adult features.

    Two learnable layers (each a ``nn.Linear`` with its bias folded into the
    weight, the project's bias trick). The hidden ReLU gives the model the
    non-linearity it needs to beat a plain logistic regression; the final layer
    produces two logits, one per income class.
    """

    def __init__(self, n_features: int, hidden: int = 64) -> None:
        self.fc1 = nn.Linear(n_features, hidden)
        self.fc2 = nn.Linear(hidden, 2)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        # x: (n_features, batch)  ->  logits: (2, batch), column-oriented.
        h = self.fc1(x).relu()
        return self.fc2(h)

## 3. Two outputs, not one number

The last layer emits **two logits per person**, $\mathbf{z} = (z_1, z_2)$ — raw scores,
free to be any real number. Softmax turns them into a probability distribution over the
two classes, and cross entropy scores that distribution against the truth:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}},
\qquad\qquad
L = -\frac{1}{N}\sum_{i=1}^{N} \log p_{i, y_i}$$

Only the probability given to the *correct* class enters the sum: the loss is small when
the model was confident and right, and grows without bound as it becomes confident and
wrong. This is Lecture I, *Softmax outputs and categorical cross entropy*.

Framing a yes/no question as **2 classes** (instead of one sigmoid output) is what lets us
reuse `cross_entropy` unchanged — and it is the same machinery a 30 000-token vocabulary
will use later.

One practical wrinkle you will see in the code: our engine is column-oriented
(`logits` is `(2, batch)`) while `cross_entropy` wants the class axis **last**. Hence the
transpose in `cross_entropy(model(X).T, y)`.

### What makes it trainable

Everything hangs on one gradient. For softmax + cross entropy together,

$$\frac{\partial L}{\partial \mathbf{Z}} = \frac{\mathrm{softmax}(\mathbf{Z}) - \mathbf{Y}_{\text{one-hot}}}{N}$$

— "the probability you gave minus the probability you should have given", divided by the
batch size (Lecture I, *Softmax + Cross Entropy: A Useful Gradient*). It is remarkably
simple for something that goes through an exponential, a normalisation and a logarithm.

The cell below is not part of the training; it checks that claim. Three hand-written
logits, three labels, one `backward()`, and we compare the `.grad` the engine produced
with the formula computed by hand in NumPy.

In [ ]:
logits = cpu.Tensor(np.array([[2.0, 1.0],      # sample 0: leans class 0
                              [0.5, 3.0],      # sample 1: leans class 1
                              [1.0, 1.0]]))    # sample 2: undecided
labels = np.array([0, 1, 1])

loss = cross_entropy(logits, labels)           # class axis is last: (N, 2)
loss.backward()

# The formula, by hand: (softmax - one_hot) / N
e = np.exp(logits.data - logits.data.max(axis=1, keepdims=True))
p = e / e.sum(axis=1, keepdims=True)
one_hot = np.zeros_like(p)
one_hot[np.arange(len(labels)), labels] = 1.0
by_hand = (p - one_hot) / len(labels)

print("loss                =", float(loss.data))
print("softmax p           =\n", p)
print("engine  dL/dlogits  =\n", logits.grad)
print("formula dL/dlogits  =\n", by_hand)
print("\nmax |engine - formula| =", float(np.abs(logits.grad - by_hand).max()))

## 4. Training is an optimisation problem

The loss is a function of the weights, so training is the search for weights that make it
small (Lecture I, *Training as an Optimization Problem* and *Gradient Descent*). Gradient
descent walks downhill: compute $\partial L/\partial \mathbf{W}$ for every weight, take a
step against it, repeat.

In code that is always the same four lines:

```python
opt.zero_grad()                          # clear last step's gradients
loss = cross_entropy(model(Xt).T, y)     # forward: builds the graph
loss.backward()                          # one graph -> every weight's .grad
opt.step()                               # W -= lr * (update)
```

Two choices worth naming:

- **Full batch.** The whole training split fits in memory, so each epoch is *one* step
  computed over all 26 049 rows — no mini-batches, no sampling noise. (Ex. 06 does the
  opposite, because there the pairs number in the hundreds of thousands.)
- **Adam instead of plain descent.** Adam keeps two running averages per parameter: `m`,
  a smoothed gradient (momentum), and `v`, a smoothed *squared* gradient (how big the
  gradient has been lately). The step divides by $\sqrt{v}$, so coordinates with wildly
  different gradient scales still move sensibly — see `bert_cpu/optim.py` for the full
  rule.

### Validation: the number that tells you when to stop

We hold out 20 % of the training base and, after every epoch, measure the loss there
**with a forward pass only** — no backward, no step, so the model never learns from it.
Read the two curves together:

| what you see | what it means |
|---|---|
| both falling | the model is still learning |
| train falling, validation flat | it is running out of things to generalise |
| train falling, **validation turning back up** | over-fitting: it is memorising rows |

That last shape is the one to recognise; the plot at the end of this notebook draws it.

In [ ]:
def accuracy(model: AdultMLP, X: np.ndarray, y: np.ndarray) -> float:
    """Fraction of samples whose arg-max logit matches the label.

    A pure forward pass (no graph needed for a metric); ``model`` predicts the
    class with the larger logit for every column of ``X``.
    """
    logits = model(cpu.Tensor(X)).data          # (2, n_samples)
    preds = logits.argmax(axis=0)               # class per sample
    return float((preds == y).mean())


def train_val_split(X: np.ndarray, y: np.ndarray, val_frac: float = 0.2):
    """Carve a validation set out of the training base (column-oriented split).

    Shuffles the sample indices once and holds out ``val_frac`` of them for
    validation. Returns ``(X_tr, y_tr, X_val, y_val)``.
    """
    n = X.shape[1]                              # samples across axis 1
    perm = np.random.permutation(n)
    n_val = int(n * val_frac)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    return X[:, tr_idx], y[tr_idx], X[:, val_idx], y[val_idx]

In [ ]:
def train(
    model: AdultMLP,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 100,
    lr: float = 1e-2,
):
    """Full-batch training with Adam; report train and validation loss per epoch.

    The whole training split is one batch, so each epoch is a single
    forward/backward over every row. The input tensors are built once and marked
    ``requires_grad=False`` (we need gradients on the *parameters*, not the data),
    which also avoids uselessly accumulating gradient into the inputs.

    Returns the two per-epoch loss histories, so they can be plotted afterwards.
    """
    opt = optim.Adam(model.parameters(), lr=lr)
    Xt = cpu.Tensor(X_tr, requires_grad=False)      # (n_features, n_train), a constant
    Xv = cpu.Tensor(X_val, requires_grad=False)     # (n_features, n_val),   a constant

    train_hist, val_hist = [], []
    total_flops = 0
    for epoch in range(1, epochs + 1):
        cpu.reset_flops()                           # start this epoch's FLOP tally

        opt.zero_grad()
        loss = cross_entropy(model(Xt).T, y_tr)     # full-batch training loss
        loss.backward()                             # one graph -> every weight's grad
        opt.step()

        # Validation loss: a forward pass only (no backward, no step) on the
        # held-out slice of the training base.
        val_loss = float(cross_entropy(model(Xv).T, y_val).data)

        # FLOPs the engine executed this epoch (train forward + backward + the
        # validation forward). It is the same every epoch — the graph is fixed.
        epoch_flops = cpu.flop_count()
        total_flops += epoch_flops

        train_hist.append(float(loss.data))
        val_hist.append(val_loss)

        print(f"  epoch {epoch:3d}/{epochs}   train loss = {float(loss.data):.4f}"
              f"   val loss = {val_loss:.4f}   FLOPs = {epoch_flops:,}")

    print(f"\nTotal FLOPs over {epochs} epochs: {total_flops:,}"
          f"   (~{total_flops / 1e9:.2f} GFLOP)")
    return train_hist, val_hist

## 5. Run it

Four cells, in the order the pipeline runs: split, build, train, measure. Keep them in
this order — the split and the weight initialisation both draw from the seed set at the
top, so running them out of order changes the numbers.

In [ ]:
# Hold out 20% of the training base for validation.
X_tr, y_tr, X_val, y_val = train_val_split(train_ds.X, train_ds.y, val_frac=0.2)
print(f"Train/val split: {X_tr.shape[1]} train / {X_val.shape[1]} val (20%)\n")

In [ ]:
model = AdultMLP(train_ds.n_features, hidden=64)
print(f"Model: Linear({train_ds.n_features}, 64) -> ReLU -> Linear(64, 2)")
print(f"Trainable parameter tensors: {len(model.parameters())}")

# Two matrices, bias rows included: (108+1)x64 + (64+1)x2.
n_params = sum(p.data.size for p in model.parameters())
print(f"Learnable numbers in total: {n_params:,}"
      f"   = {(train_ds.n_features + 1) * 64:,} + {(64 + 1) * 2}\n")

In [ ]:
print("Training (full-batch Adam):")
train_hist, val_hist = train(model, X_tr, y_tr, X_val, y_val, epochs=100, lr=1e-2)

In [ ]:
train_acc = accuracy(model, X_tr, y_tr)
val_acc = accuracy(model, X_val, y_val)
test_acc = accuracy(model, test_ds.X, test_ds.y)
print(f"\nFinal accuracy   train = {train_acc:.4f}   val = {val_acc:.4f}   test = {test_acc:.4f}")

## 6. Read the result

Three things to take away from those numbers:

1. **It beat the baseline.** Test accuracy lands around 0.85 against the 0.76 of "always
   answer `<=50K`" — the network learned something real about the census features, not
   just the class proportions.
2. **Train is above validation and test.** The gap (≈ 0.87 vs ≈ 0.85) is the model fitting
   the training rows slightly better than the world. Small here; the plot below shows
   whether it was still growing when we stopped.
3. **The compute is the matmuls.** The FLOP tally — around 84 GFLOP for 100 epochs, so
   842 MFLOP per epoch — counts what the engine actually executed, and it splits roughly in
   half: ~372 MFLOP for the training forward pass, ~377 MFLOP for the backward, ~93 MFLOP
   for the validation forward on the smaller held-out slice. Almost all of it is the first
   layer's matrix product, $\mathbf{Z}^{(1)} = (\mathbf{W}^{(1)})^{\top}\widetilde{\mathbf{X}}$
   — 2 · (64 · 26 049) · 109 ≈ 363 MFLOP on its own (Lecture I, *Matrix Form of
   Backpropagation in a Dense Layer*). Put `cpu.reset_flops()` before a forward pass and
   `cpu.flop_count()` after it to split the tally yourself.

### The two curves (optional — needs matplotlib)

`pip install matplotlib` if the cell reports it missing; the library itself stays
NumPy-only.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed — run `pip install matplotlib` to see the curves.")
else:
    epochs = range(1, len(train_hist) + 1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, train_hist, label="train loss")
    ax.plot(epochs, val_hist, "--", label="validation loss")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cross entropy")
    ax.set_title("Adult: training vs validation loss")
    ax.legend()
    fig.tight_layout()
    plt.show()

---

**Next:** [Exercise 06 — learning word embeddings](q06_learn_embedding.ipynb). Same four
lines of training, but the data is raw text and nobody hands us labels: the corpus has to
generate its own.

And that is the whole trick of what follows in this repository — BERT is this same cycle
(forward, one scalar loss, `backward()`, `step()`) on a much bigger graph.